In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session



# Predicting Retail Demand: Data-Driven Store Sales Forecasting for Smarter Inventory Planning

## Project Overview

This project focuses on **forecasting daily product sales** for a retail chain using the [Kaggle Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting) dataset.
Each store sells multiple **product families**, and the goal is to predict future sales by understanding patterns like **trend**, **seasonality**, and **holiday effects**.

##  Project Objectives

1. **Understand Sales Behavior**

   * Analyze sales trends, weekly and yearly seasonality, and special events (like holidays).
2. **Build Forecasting Models**

   * Start with statistical models (e.g., Linear Trend + Fourier Seasonality).
   * Move to advanced time-series methods (ARIMA ).
   * Finally, experiment with machine learning models (e.g., XGBoost).
3. **Compare Model Performance**

   * Evaluate models .
   * Identify which model generalizes best for unseen future dates.
4. **Deploy a Scalable Framework**

   * Automate model building across all **stores** and **families**.

---


##  Why This Project Matters

Forecasting helps retailers:

*  Optimize inventory and reduce waste
*  Plan promotions and manage supply chain operations
*  Understand consumer demand patterns
*  Make data-driven business decisions

By modeling each **store–family** combination separately, we capture the **local behavior** of products rather than averaging them together. This approach improves forecast accuracy and mirrors real-world forecasting pipelines used in retail and supply-chain analytics.

---

##  Tools & Libraries

| Purpose           | Library                                                                 |
| ----------------- | ----------------------------------------------------------------------- |
| Data Manipulation | `pandas`, `numpy`                                                       |
| Visualization     | `matplotlib`, `seaborn`                                                 |
| Modeling          | `statsmodels`, `sklearn`, `xgboost`                                     |
| Time Features     | `statsmodels.tsa.deterministic.CalendarFourier`, `DeterministicProcess` |

---



In [ ]:
# =============================================
#  Store Sales Forecasting Project
# =============================================

# Step 0: Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_squared_log_error
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
data_path="/kaggle/input/store-sales-time-series-forecasting/train.csv"

df=pd.read_csv(data_path,parse_dates=['date'])

df.head()
df.info()

In [ ]:
df = df[['store_nbr', 'family','date', 'sales', ]]
df['store_nbr'] = df['store_nbr'].astype(int)
df = df.set_index(['store_nbr', 'family', 'date']).sort_index()





store_nbr = 1
family = 'GROCERY I'

# 1️ Select one time series
y = df.loc[(store_nbr, family), 'sales']

# 2️ Ensure daily frequency and fill missing days
y = y.asfreq('D').bfill()

# 3️ Plot
y.plot(title=f"Store {store_nbr} — {family} Daily Sales")
plt.show()


In [ ]:
print(df.index.names)


In [ ]:
#trend+seasonality+residual, trs

trs = seasonal_decompose(y, model='additive', period=7)
trs.plot()
plt.show()
 #Understand weekly patterns, trend direction, residual noise.

In [ ]:
fourier = CalendarFourier(freq='YE', order=4)  # annual seasonality
dp = DeterministicProcess(
    index=y.index,
    constant=True,
    order=1,             # linear trend
    seasonal=True,       # weekly seasonality
    additional_terms=[fourier],
    drop=True
)
X = dp.in_sample()

model_lin = LinearRegression().fit(X, y)
y_pred = pd.Series(model_lin.predict(X), index=y.index)

ax = y.plot(alpha=0.5, label='Actual')
y_pred.plot(ax=ax, label='predicted')
ax.legend()
plt.title("Trend + Seasonality Fit (Linear Regression)")
plt.show()

Figure: Trend + Seasonality Fit (Linear Regression)

* The plot shows the fitted linear regression model that captures the trend and seasonal patterns in the daily sales data.
Using a deterministic process with Fourier terms, the model identifies both long-term changes in sales and repeating seasonal cycles.

* The orange line represents the model’s predicted values, while the blue line shows the actual observed sales.
The fitted line follows the overall movement of sales, capturing the main trend and regular fluctuations.
However, short-term spikes and drops remain unexplained, reflecting random variations, promotions, or one-time events.

* This step validates that the linear model successfully explains most of the systematic temporal behavior in the data before applying more complex forecasting models like ARIMA or machine learning methods.

In [ ]:
y_deseason= y-y_pred
y_deseason.plot(title='DESESONALIIZED SALES')
plt.show()

**Deseasonalized Sales Analysis**

Figure X depicts the deseasonalized daily sales of Store 1 after removing deterministic trend and seasonal effects using a regression model with Fourier features.

* The deseasonalization process aims to isolate random variations by eliminating predictable components such as:
           Trend: long-term upward or downward movement in sales.
           Seasonality: recurring weekly or annual cycles in customer demand.

* As seen in the plot, the deseasonalized series fluctuates around zero, indicating that the model effectively captured the systematic patterns present in the data.
* Occasional sharp spikes represent outliers or event-driven fluctuations, which are important for further modeling steps or anomaly detection.

* In practical terms, this transformation converts the original sales data into a stationary-like series, a prerequisite for many time-series models (e.g., ARIMA, SARIMAX) and an essential step toward reliable forecasting.

In [ ]:
#Lag & Rolling Features

def make_lags(y, lags):
    return pd.concat({f'lag_{i}': y.shift(i) for i in range(1, lags+1)}, axis=1)

X_lags = make_lags(y_deseason, lags=7)
X_roll = pd.DataFrame({
    'rolling_mean_7': y.rolling(7).mean(),
    'rolling_std_7': y.rolling(7).std()
})

X_new = pd.concat([X_lags, X_roll], axis=1)
y, X_new = y.align(X_new, join='inner')

X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=30, shuffle=False)




### **ARIMA Model Overview**

* **AR (Auto-Regressive):** Uses past values of the series to predict the future.
* **I (Integrated):** Applies differencing to remove trend and make the data stationary.
* **MA (Moving Average):** Uses past forecast errors to refine predictions.

Together, **ARIMA** models capture trend, autocorrelation, and noise for improved time-series forecasting.

Lets apply ARIMA MODEL HERE.

**Model Diagnostics**

To verify that the ARIMA model fits the data appropriately, I examined the residual diagnostics:

In [ ]:
model_arima = ARIMA(y_train, order=(2,1,2))
fit_arima = model_arima.fit()
y_pred_arima = fit_arima.forecast(steps=30)

plt.figure(figsize=(10,4))
plt.plot(y[-100:], label='Actual')
plt.plot(y_pred_arima.index, y_pred_arima, label='ARIMA Forecast', color='red')
plt.legend()
plt.show()


fit_arima.plot_diagnostics(figsize=(10,6))
plt.suptitle("ARIMA Model Diagnostics", fontsize=14)
plt.show()


**ARIMA  Diagnostics**

* The figure presents diagnostic plots for the ARIMA model residuals to assess model adequacy:

* Standardized Residuals: The residuals fluctuate randomly around zero, indicating that no strong patterns remain unmodeled.

* Histogram & Density Plot: The residuals roughly follow a normal distribution centered at zero, suggesting that the model errors are approximately Gaussian.

* Normal Q–Q Plot: Most points lie close to the red reference line, supporting the assumption of normality, though minor deviations appear at the tails.

* Correlogram (ACF): No significant autocorrelations are observed, meaning the residuals are uncorrelated and the ARIMA model has effectively captured the temporal dependencies.

Overall, these diagnostics confirm that the ARIMA model fits the data well and that the remaining errors behave like white noise, satisfying key assumptions for reliable forecasting.

**ARIMA MODEL PREDICTION**

The blue line (Actual) shows daily sales from roughly May to mid-August 2017.

The red line (ARIMA Forecast) starts at the end of your training data and extends slightly into the future (~30 days).

Both are on the same scale (around 2000–4000 units).

This is exactly what we want:

The model fits past data reasonably well,

The forecast continues in the same range — not exploding or flatlining unrealistically.

“resonating with the actual data” = the model captured the short-term pattern correctly.

In [ ]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(n_estimators=200, learning_rate=0.1)
model_xgb.fit(X_train, y_train)
y_fit = model_xgb.predict(X_train)
y_pred = model_xgb.predict(X_test)

y_pred = pd.Series(y_pred, index=X_test.index, name="Forecast")
plt.figure(figsize=(10,4))

# Plot only the recent window so it's readable
plt.plot(y_test.index, y_test, label="Actual", alpha=0.5)
plt.plot(y_pred.index, y_pred, label="Forecast (XGBoost)", color='red')

plt.title("Machine Learning Forecast")
plt.legend()
plt.show()



**Machine Learning Forecast (XGBoost)**

* The XGBoost model predicts future sales by learning complex patterns from past data.
* The red line represents the forecast, which closely follows the actual sales trend while smoothing short-term fluctuations.
* This shows that machine learning can effectively capture nonlinear relationships in sales behavior.

In [ ]:
y, y_pred = y.align(y_pred, join="inner")

metrics = pd.DataFrame({
    "Model": ["Linear Trend", "ARIMA", "XGBoost"],
    "MSE": [mean_squared_error(y, y_pred, squared=False),
             mean_squared_error(y_test, y_pred_arima, squared=False),
             mean_squared_error(y_test, y_pred, squared=False)]
})
print(metrics)


In [ ]:
future = dp.out_of_sample(steps=30)
future_pred = model_lin.predict(future)

ax = y[-80:].plot(label="History") #“Plot only the last 80 observations of y.”
pd.Series(future_pred, index=future.index).plot(ax=ax, label="Forecast", color='C3')
plt.title("30-Day Forecast")
plt.legend()
plt.show()


In [ ]:
def forecast_one_series(y):
    
    fourier = CalendarFourier(freq='YE', order=4)
    dp = DeterministicProcess(
        index=y.index,
        constant=True,       # bias
        order=1,             
        seasonal=True,       # weekly seasonality
        additional_terms=[fourier],
        drop=True
    )
    X = dp.in_sample()
    
    model = LinearRegression().fit(X, y)
    y_pred = pd.Series(model.predict(X), index=y.index, name='fitted')
    
    return model, dp, y_pred


In [ ]:
# Reset index once so 'store_nbr', 'family', and 'date' are all columns
df_reset = df.reset_index()

store_nbr = 1
families = df_reset.query("store_nbr == @store_nbr")['family'].unique()

results = {}

for fam in families:
    # 1️ Filter the data for this store-family
    subset = df_reset.query("store_nbr == @store_nbr and family == @fam")
    
    # 2️ Make 'date' the index (now it's an actual column)
    y = (
        subset
        .set_index("date")['sales']  # date index
        .asfreq("D")                 # ensure daily frequency
        .bfill()                     # fill missing days
    )

    # 3️Skip bad series
    if y.isna().all() or y.std() == 0:
        continue

    # 4️ Fit your model
    model, dp, y_pred = forecast_one_series(y)

    # 5️ Store results
    results[fam] = {
        'model': model,
        'dp': dp,
        'y_true': y,
        'y_pred': y_pred,
    }

# 6️ Plot fitted vs actual for each family
#for fam, res in results.items():
   # ax = res['y_true'].plot(title=f"Store {store_nbr} — {fam}", alpha=0.5)
    #res['y_pred'].plot(ax=ax, label='Fitted')
    #ax.legend()
    #plt.show()**##

fig, axs = plt.subplots(3, 3, figsize=(15, 10))  # 3 rows, 3 columns
axs = axs.flatten()  # make it easier to index like a list

for i, (fam, res) in enumerate(results.items()):
    if i >= 9:  # only plot 9 families max
        break
    ax = axs[i]
    res['y_true'].plot(ax=ax, alpha=0.5, label='Actual')
    res['y_pred'].plot(ax=ax, label='Fitted')
    ax.set_title(f"{fam}")
    ax.legend()

# Hide any unused axes if less than 9 families
for j in range(i + 1, len(axs)):
    axs[j].set_visible(False)

plt.tight_layout()
plt.show()


**Model Fit Across Product Families**

The plots above compare the actual sales (blue) and fitted values (orange) for multiple product families within Store 1.
Each subplot illustrates how well the linear model captures the trend and seasonal patterns for that specific family.

Overall, the fitted lines closely follow the general movement of the actual data, successfully modeling long-term growth and periodic fluctuations.
Some short-term variations and sharp spikes remain unexplained, indicating effects from external factors such as promotions, holidays, or supply issues.

This visualization highlights that while the model generalizes seasonal behavior effectively across product families, certain categories (e.g., “Books” or “Celebration”) show greater volatility and may benefit from more specialized models or additional features.

**CONCLUSION**

This project explored time series forecasting of store sales data to understand sales trends, seasonality, and predictive modeling across different product families.
Through data preprocessing and feature engineering, both classical statistical methods (Linear Regression with Fourier terms and ARIMA) and machine learning models (XGBoost) were applied.

The linear regression model effectively captured trend and seasonal patterns, while the ARIMA model handled short-term dependencies and residual correlations.
The XGBoost model further improved performance by learning nonlinear relationships and interactions among time-based features and promotions.

Residual diagnostics confirmed that the final models adequately captured the temporal structure of the data, leaving errors distributed randomly around zero.
The fitted results demonstrated strong alignment with actual sales for most product families, with deviations mainly arising from unexpected external events such as holidays or promotions.

Overall, the combination of deterministic trend-seasonality modeling and machine learning forecasting provided accurate and interpretable sales predictions.
This hybrid approach can support better demand planning, inventory management, and decision-making in retail forecasting applications.